<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br><br><br>

# Listed Volatility and Variance Derivatives

**Wiley Finance (2017)**

Dr. Yves J. Hilpisch | The Python Quants GmbH

http://tpq.io | [@dyjh](http://twitter.com/dyjh) | http://books.tpq.io

<img src="https://hilpisch.com/images/lvvd_cover.png" alt="Listed Volatility and Variance Derivatives" width="30%" align="left" border="0">

# DX Analytics &mdash; Square-Root Diffusion 

## Introduction 

This chapter uses DX Analytics to model the VSTOXX volatility index by a square-root diffusion process as proposed in Gruenbichler and Longstaff (1996) and discussed in chapter _Valuing Volatility Derivatives_. It implements a study over a time period of three months to analyze how well the model performs in replicating market quotes for VSTOXX options.

## Data Import and Selection

The data we are working with is for the first quarter of 2014. The complete data set is contained in the online resources accompanying this book. As usual, some imports first.

In [1]:
import numpy as np
import pandas as pd
import datetime as dt

Next, we read the data from the source into pandas ``DataFrame`` objects.

In [2]:
h5 = pd.HDFStore('./data/vstoxx_march_2014.h5', 'r')
vstoxx_index = h5['vstoxx_index']  # data for the index itself 
vstoxx_futures = h5['vstoxx_futures']  # data for the futures
vstoxx_options = h5['vstoxx_options']  # data for the options
h5.close()

Inspecting the data sub-set for the VSTOXX index itself, we see that we are dealing with 63 trading days.

In [3]:
vstoxx_index.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 63 entries, 2014-01-02 to 2014-03-31
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   V2TX    63 non-null     float64
 1   V6I1    57 non-null     float64
 2   V6I2    63 non-null     float64
 3   V6I3    61 non-null     float64
 4   V6I4    63 non-null     float64
 5   V6I5    63 non-null     float64
 6   V6I6    62 non-null     float64
 7   V6I7    63 non-null     float64
 8   V6I8    63 non-null     float64
dtypes: float64(9)
memory usage: 4.9 KB


In [4]:
vstoxx_index.tail()

,V2TX,V6I1,V6I2,V6I3,V6I4,V6I5,V6I6,V6I7,V6I8
Date,,,,,,,,,
2014-03-25,18.2637,18.2303,18.3078,19.0371,19.8378,20.3065,18.1063,20.8292,21.2046
2014-03-26,17.5869,17.4810,17.7009,18.4499,19.4150,19.9961,20.2562,20.4541,20.8563
2014-03-27,17.6397,17.5032,17.7608,18.6249,19.4860,20.0477,20.1078,20.4865,20.9449
2014-03-28,17.0324,16.6849,17.2864,18.3281,19.3032,19.8332,20.1371,20.3808,20.8210
2014-03-31,17.6639,17.6087,17.6879,18.5689,19.4285,20.0430,19.9823,20.4448,20.8994


Per trading day, there are eight futures quotes for the eight different maturities of the VSTOXX futures contract. This makes for a total of 504 futures quotes.

In [5]:
vstoxx_futures['DATE'] = pd.to_datetime(vstoxx_futures['DATE'])

In [6]:
vstoxx_futures['MATURITY'] = pd.to_datetime(vstoxx_futures['MATURITY'])

In [7]:
vstoxx_futures.info()

<class 'pandas.DataFrame'>
Index: 504 entries, 0 to 503
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   DATE       504 non-null    datetime64[ns]
 1   EXP_YEAR   504 non-null    int64         
 2   EXP_MONTH  504 non-null    int64         
 3   PRICE      504 non-null    float64       
 4   MATURITY   504 non-null    datetime64[ns]
dtypes: datetime64[ns](2), float64(1), int64(2)
memory usage: 23.6 KB


In [8]:
vstoxx_futures.tail()

,DATE,EXP_YEAR,EXP_MONTH,PRICE,MATURITY
499,2014-03-31,2014,7,20.40,2014-07-18
500,2014-03-31,2014,8,20.70,2014-08-15
501,2014-03-31,2014,9,20.95,2014-09-19
502,2014-03-31,2014,10,21.05,2014-10-17
503,2014-03-31,2014,11,21.25,2014-11-21


By far the biggest data sub-set is the one for the VSTOXX options. There are for each trading day market quotes for puts and calls for eight different maturities and a multitude of different strike prices. This makes for a total of 46960 option quotes for the first quarter of 2014.

In [9]:
vstoxx_options['DATE'] = pd.to_datetime(vstoxx_options['DATE'])

In [10]:
vstoxx_options['MATURITY'] = pd.to_datetime(vstoxx_options['MATURITY'])

In [11]:
vstoxx_options.info()

<class 'pandas.DataFrame'>
Index: 46960 entries, 0 to 46959
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   DATE       46960 non-null  datetime64[ns]
 1   EXP_YEAR   46960 non-null  int64         
 2   EXP_MONTH  46960 non-null  int64         
 3   TYPE       46960 non-null  str           
 4   STRIKE     46960 non-null  float64       
 5   PRICE      46960 non-null  float64       
 6   MATURITY   46960 non-null  datetime64[ns]
dtypes: datetime64[ns](2), float64(2), int64(2), str(1)
memory usage: 2.9 MB


In [12]:
vstoxx_options.tail()

,DATE,EXP_YEAR,EXP_MONTH,TYPE,STRIKE,PRICE,MATURITY
46955,2014-03-31,2014,11,P,85.0,63.65,2014-11-21
46956,2014-03-31,2014,11,P,90.0,68.65,2014-11-21
46957,2014-03-31,2014,11,P,95.0,73.65,2014-11-21
46958,2014-03-31,2014,11,P,100.0,78.65,2014-11-21
46959,2014-03-31,2014,11,P,105.0,83.65,2014-11-21


Maturity-wise we are dealing with a total of eleven dates. This is due to the fact that at any given time eight maturities for the VSTOXX futures and options contracts are available and we are looking at data for three months.

In [13]:
third_fridays = sorted(set(vstoxx_futures['MATURITY']))
third_fridays

[Timestamp('2014-01-17 00:00:00'),
 Timestamp('2014-02-21 00:00:00'),
 Timestamp('2014-03-21 00:00:00'),
 Timestamp('2014-04-18 00:00:00'),
 Timestamp('2014-05-16 00:00:00'),
 Timestamp('2014-06-20 00:00:00'),
 Timestamp('2014-07-18 00:00:00'),
 Timestamp('2014-08-15 00:00:00'),
 Timestamp('2014-09-19 00:00:00'),
 Timestamp('2014-10-17 00:00:00'),
 Timestamp('2014-11-21 00:00:00')]

When it comes to the calibration of the square-root diffusion model for the VSTOXX, it is necessary to work with a selection from the large set of option quotes. The following function implements such a selection procedure, using different conditions to generate a sensible set of option quotes around the forward at-the-money level. The function `srd_get_option_selection()` is used in what follows to select the right sub-set of option quotes for each day during the calibration.

In [14]:
import sys
sys.path.append('scripts')

**Note (2026-09-05):** This notebook cannot run as-is. It depends on `dx` — The Python Quants' proprietary **DX Analytics** library for derivatives valuation and calibration. It is not the unrelated `dx` package on PyPI (that's `nteract/dx`, a Jupyter display library) and it is not installed in this environment or bundled in this repository. The cells below are kept as reference from the original book material; running them requires obtaining and installing DX Analytics separately.

In [15]:
import dx_srd_calibration as dxsrd

In [16]:
dxsrd.srd_get_option_selection??

## Modeling the VSTOXX Options

The previous chapter illustrates how European options are modeled with DX Analytics based on a geometric Brownian motion model (`dx.geometric_brownian_motion()`). To model the VSTOXX options for the calibration, we just need to replace that model by the square-root diffusion model `dx.square_root_diffusion()`. The respective market environment then needs some additional parameters.

All the code used for the calibration is found in the Python script ``dx_srd_calibration.py`` (see the appendix for the complete script). After some imports, the script starts by defining some general parameters and curves for the market environment. During the calibration process, some of these get updated to reflect the current status of the optimization procedure. 

In [17]:
!sed -n 11,53p scripts/dx_srd_calibration.py

import dx
import time
import numpy as np
import pandas as pd
import datetime as dt
import scipy.optimize as spo
from pylab import mpl, plt
plt.style.use('seaborn-v0_8')
mpl.rcParams['font.family'] = 'serif'

# importing the data
h5 = pd.HDFStore('./data/vstoxx_march_2014.h5', 'r')
vstoxx_index = h5['vstoxx_index']
vstoxx_futures = h5['vstoxx_futures']
vstoxx_options = h5['vstoxx_options']
h5.close()
vstoxx_futures['DATE'] = pd.to_datetime(vstoxx_futures['DATE'])
vstoxx_futures['MATURITY'] = pd.to_datetime(vstoxx_futures['MATURITY'])
vstoxx_options['DATE'] = pd.to_datetime(vstoxx_options['DATE'])
vstoxx_options['MATURITY'] = pd.to_datetime(vstoxx_options['MATURITY'])
# collecting the maturity dates
third_fridays = sorted(set(vstoxx_futures['MATURITY']))

# instantiation of market environment object with dummy pricing date
me_vstoxx = dx.market_environment('me_vstoxx', dt.datetime(2014, 1, 1))
me_vstoxx.add_constant('currency', 'EUR')
me_vstoxx.add_constant('frequency', 'W')
me_vstoxx.ad

The function `srd_get_option_models()` creates valuation models for all options in a given selection of option quotes.

In [18]:
dxsrd.srd_get_option_models??

## Calibration of the VSTOXX Model

Calibration of a parametrized model usually boils down to using global and local optimization algorithms to find parameters that minimize a given target function. This process is discussed in detail in Hilpisch (2015, ch. 11). For the calibration process to follow, we use the helper function `srd_calculate_model_values()` to calculate "at once" the model values for the VSTOXX options at hand. The function parameter `p0` is a tuple since this is what the optimization functions provide as input.

In [19]:
dxsrd.srd_calculate_model_values??

The target function to be minimized during the calibration is the _mean-squared error_ of the model values given the market quotes of the VSTOXX options. Again, refer to Hilpisch (2015, ch. 11) for details and alternative formulations. The function `srd_mean_squared_error()` implements this concept and uses the function `srd_calculate_model_values()` for the option model value calculations.

In [20]:
dxsrd.srd_mean_squared_error??

Equipped with the target function to be minimized, we can define the function for the global and local calibration routine itself. The calibration takes place for one or multiple maturities over the pricing date range defined. For example, the function `srd_get_parameter_series()` can calibrate the model (separately) for the two maturities May and June 2014 over the complete first quarter 2014.

In [21]:
dxsrd.srd_get_parameter_series??

The final step is to start the calibration and collect the calibration results. The calibration we implement is for the 18. April 2014 maturity.

In [22]:
!sed -n 256,287p scripts/dx_srd_calibration.py

if __name__ == '__main__':
    t0 = time.time()
    # define the dates for the calibration
    pricing_date_list = pd.date_range('2014/1/2', '2014/3/31', freq='B')
    # select the maturities
    maturity_list = [third_fridays[3]]  # only 18. April 2014 maturity
    # start the calibration
    parameters = srd_get_parameter_series(pricing_date_list, maturity_list)
    # plot the results
    for mat in maturity_list:
        # fig1, ax1 = plt.subplots()
        to_plot = parameters[parameters.maturity ==
                         maturity_list[0]].set_index('date')[
                        ['kappa', 'theta', 'sigma', 'MSE']]
        to_plot.plot(subplots=True, color='b', figsize=(10, 12),
                 title='SRD | ' + str(mat)[:10])
        plt.tight_layout()
        plt.savefig('./images/dx_srd_cali_1_%s_.png' % str(mat)[:10])
        # plotting the histogram of the MSE values
        fig, ax = plt.subplots()
        dat = parameters.MSE
        dat.hist(bins=30, ax=ax)
        plt.

### No Penalization

A visualization of the calibration results tells the whole story. The following figures shows the three square-root diffusion parameters over time and the resulting MSE values. 

In [23]:
# %run scripts/dx_srd_calibration.py

<img src="./images/dx_srd_cali_1_2014-04-18.png" width="75%">

<p style="font-family: monospace;">Square-root diffusion parameters and MSE values from the calibration to a single maturity (18. April 2014) &mdash; no penalization.

As we can see throughout, the results are quite good given the low MSE values. The mean MSE value is below 0.01 as seen in the following figure.

<img src="images/dx_srd_cali_1_hist_2014-04-18.png" width="75%">
 
<p style="font-family: monospace;">Histogram of the mean-squared errors for the calibration of the square-root diffusion model to a single maturity (18. April 2015) &mdash; no penalization.

### With Penalization

<img src="./images/dx_srd_cali_1_2014-04-18_.png" width="75%">

<p style="font-family: monospace;">Square-root diffusion parameters and MSE values from the calibration to a single maturity (18. April 2014) &mdash; with penalization.

As we can see throughout, the results are quite good given the low MSE values. The mean MSE value is below 0.01 as seen in the following figure.

<img src="images/dx_srd_cali_1_hist_2014-04-18_.png" width="75%">
 
<p style="font-family: monospace;">Histogram of the mean-squared errors for the calibration of the square-root diffusion model to a single maturity (18. April 2015) &mdash; with penalization.

## Conclusions

This chapter uses DX Analytics to model the VSTOXX volatility index by a square-root diffusion process. In similar vein, it is used to model traded European call options on the index to implement a calibration of the VSTOXX model over time. The results show that when calibrating the model to a single options maturity only, the model performs quite well yielding rather low MSE values throughout. The parameter values also seem to be in sensible regions throughout (e.g. `theta` between 15 and 18) and they evolve rather smoothly.

There exist closed form solutions for the price of a European call option in the square-root diffusion model of Gruenbichler and Longstaff (1996) as shown in chapter _Valuing Volatility Derivatives_. For our analysis in this chapter we have nevertheless used the Monte Carlo valuation model of DX Analytics since this approach is more general in that we can easily replace one model by another, maybe more sophisticated, one. This is done in the next chapter where the same study is implemented based on the square-root jump diffusion model presented in chapter _Advanced Modeling of the VSTOXX Index_. The only difference is that a few more parameters need to be taken care of.

## Python Scripts

### `dx_srd_calibration.py`

In [24]:
!cat scripts/dx_srd_calibration.py

#
# Calibration of Gruenbichler and Longstaff (1996)
# Square-Root Diffusion (SRD) model to
# VSTOXX call options with DX Analytics
#
# All data from www.eurexchange.com
#
# (c) Dr. Yves J. Hilpisch
# Listed Volatility and Variance Derivatives
#
import dx
import time
import numpy as np
import pandas as pd
import datetime as dt
import scipy.optimize as spo
from pylab import mpl, plt
plt.style.use('seaborn-v0_8')
mpl.rcParams['font.family'] = 'serif'

# importing the data
h5 = pd.HDFStore('./data/vstoxx_march_2014.h5', 'r')
vstoxx_index = h5['vstoxx_index']
vstoxx_futures = h5['vstoxx_futures']
vstoxx_options = h5['vstoxx_options']
h5.close()
vstoxx_futures['DATE'] = pd.to_datetime(vstoxx_futures['DATE'])
vstoxx_futures['MATURITY'] = pd.to_datetime(vstoxx_futures['MATURITY'])
vstoxx_options['DATE'] = pd.to_datetime(vstoxx_options['DATE'])
vstoxx_options['MATURITY'] = pd.to_datetime(vstoxx_options['MATURITY'])
# collecting the maturity dates
third_fridays = sorted(set(vstoxx_futures['MATU

<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

<a href="http://tpq.io" target="_blank">http://tpq.io</a> | <a href="http://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:team@tpq.io">team@tpq.io</a>